In [16]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
files = sorted(data_dir.rglob("raw_*.csv"))

print(f"Found {len(files)} asset files\n")

series_list = []

for path in files:
    asset = path.stem.replace("raw_", "")

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]

    n_negative = (s < 0).sum()
    if n_negative > 0:
        print(f"  [{asset}] {n_negative} negative price(s) detected — forward-filled")

    s = s.where(s > 0, np.nan).ffill()
    log_returns = np.log(s / s.shift(1)).rename(asset)

    series_list.append(log_returns)

print(f"\nLoaded {len(series_list)} assets")

merged = pd.concat(series_list, axis=1, sort=False)
print(f"Merged shape before dropna: {merged.shape}")
print(f"Total NaNs before dropna: {merged.isna().sum().sum()}")

merged = merged.dropna()
print(f"Merged shape after dropna:  {merged.shape}")

merged = merged.reset_index()
merged = merged.sort_values("Date").reset_index(drop=True)

print(f"\nDate range: {merged['Date'].min()} → {merged['Date'].max()}")
print(f"Assets: {[c for c in merged.columns if c != 'Date']}")

merged["Date"] = merged["Date"].dt.strftime("%Y-%m-%d")
merged.to_csv(data_dir / "all_assets_log_returns.csv", index=False)

print(f"\nSaved → {data_dir / 'all_assets_log_returns.csv'}")
print(f"Final shape: {merged.shape[0]} trading days × {merged.shape[1] - 1} assets (+1 Date column)")

merged.head()


Found 40 asset files


Loaded 40 assets
Merged shape before dropna: (3774, 40)
Total NaNs before dropna: 516
Merged shape after dropna:  (3416, 40)

Date range: 2012-05-21 00:00:00 → 2025-12-30 00:00:00
Assets: ['agg', 'bnd', 'emb', 'hyg', 'ief', 'lqd', 'mub', 'shy', 'tip', 'tlt', 'coffee', 'copper', 'corn', 'crude_oil', 'gold', 'natural_gas', 'platinum', 'silver', 'soybeans', 'wheat', 'aapl', 'amzn', 'brk_b', 'cost', 'dis', 'googl', 'hd', 'intc', 'jnj', 'jpm', 'ma', 'meta', 'msft', 'nflx', 'nvda', 'pg', 'tsla', 'unh', 'v', 'xom']

Saved → ../data/all_assets_log_returns.csv
Final shape: 3416 trading days × 40 assets (+1 Date column)


,Date,agg,bnd,emb,hyg,ief,lqd,mub,shy,tip,...,ma,meta,msft,nflx,nvda,pg,tsla,unh,v,xom
0,2012-05-21,-0.000181,0.000119,0.005965,0.010336,-0.002612,0.001126,-0.000721,0.000119,0.001327,...,0.039799,-0.116378,0.016266,0.025124,0.017234,-0.002049,0.042968,0.028485,0.031288,0.006972
1,2012-05-22,-0.002435,-0.001666,0.002969,-0.000113,-0.002151,-0.002686,-0.000632,-0.000355,-0.002074,...,0.006942,-0.093255,0.000337,-0.057814,-0.012280,-0.003793,0.068181,0.003056,0.018499,-0.001098
2,2012-05-23,0.001174,0.000714,-0.006489,-0.001017,0.002618,0.002427,0.000451,0.000473,0.000083,...,0.008790,0.031749,-0.022084,0.060182,0.024411,-0.012108,0.007118,-0.006482,0.006566,0.001098
3,2012-05-24,-0.000812,-0.001071,-0.001991,0.000226,-0.002149,0.000172,-0.000992,-0.000356,-0.001412,...,0.006547,0.031680,-0.001375,-0.023071,-0.026886,0.002881,-0.024145,0.015415,0.005022,0.006924
4,2012-05-25,0.001533,0.001191,0.001268,-0.003172,0.002896,0.002506,0.000722,0.000474,0.000997,...,-0.014080,-0.034497,-0.000344,-0.000712,0.023665,-0.001279,-0.015644,-0.001780,-0.003345,-0.006436


In [17]:
from pathlib import Path
import numpy as np
import pandas as pd

data_dir = Path("../data")
stocks_dir = data_dir / "stocks"

EXPECTED_STOCKS = [
    "aapl", "msft", "nvda", "amzn", "jpm", "jnj", "xom", "tsla", "nflx", "v",
    "googl", "meta", "brk_b", "unh", "ma", "hd", "pg", "cost", "dis", "intc",
]

# ── Load and compute log returns ──────────────────────────────────────────────
series_list = []
missing_files = []

for ticker in EXPECTED_STOCKS:
    path = stocks_dir / f"raw_{ticker}.csv"
    if not path.exists():
        print(f"  MISSING FILE: {path.name}")
        missing_files.append(ticker)
        continue

    df = pd.read_csv(path, usecols=["Date", "Close"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    s = df.set_index("Date")["Close"]
    n_negative = (s < 0).sum()
    if n_negative > 0:
        print(f"  [{ticker}] {n_negative} negative price(s) — forward-filled")
    s = s.where(s > 0, np.nan).ffill()

    log_returns = np.log(s / s.shift(1)).rename(ticker)
    series_list.append(log_returns)

# ── Merge ─────────────────────────────────────────────────────────────────────
merged = pd.concat(series_list, axis=1, sort=False)
merged = merged.reset_index().sort_values("Date").reset_index(drop=True)

# ── Validation ────────────────────────────────────────────────────────────────
print(f"\n── Stocks CSV Validation ────────────────────────────────")
print(f"  Expected : {len(EXPECTED_STOCKS)} stocks")
print(f"  Loaded   : {len(series_list)} stocks")
if missing_files:
    print(f"  MISSING  : {missing_files}")
else:
    print(f"  Missing  : none")

print(f"\n  Shape before dropna : {merged.shape[0]} rows × {merged.shape[1]-1} assets")
nan_counts = merged.drop(columns="Date").isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts):
    print(f"  NaNs per asset:\n{nan_counts.to_string()}")
else:
    print(f"  NaNs                : none")

merged_clean = merged.dropna()
print(f"  Shape after dropna  : {merged_clean.shape[0]} rows × {merged_clean.shape[1]-1} assets")
print(f"  Date range          : {merged_clean['Date'].min()} → {merged_clean['Date'].max()}")
print(f"  Stocks              : {[c for c in merged_clean.columns if c != 'Date']}")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = data_dir / "stocks_log_returns.csv"
merged_clean["Date"] = merged_clean["Date"].dt.strftime("%Y-%m-%d")
merged_clean.to_csv(out_path, index=False)
print(f"\n  Saved → {out_path}")

merged_clean.head()


── Stocks CSV Validation ────────────────────────────────
  Expected : 20 stocks
  Loaded   : 20 stocks
  Missing  : none

  Shape before dropna : 3771 rows × 20 assets
  NaNs per asset:
aapl       1
msft       1
nvda       1
amzn       1
jpm        1
jnj        1
xom        1
tsla       1
nflx       1
v          1
googl      1
meta     348
brk_b      1
unh        1
ma         1
hd         1
pg         1
cost       1
dis        1
intc       1
  Shape after dropna  : 3423 rows × 20 assets
  Date range          : 2012-05-21 00:00:00 → 2025-12-30 00:00:00
  Stocks              : ['aapl', 'msft', 'nvda', 'amzn', 'jpm', 'jnj', 'xom', 'tsla', 'nflx', 'v', 'googl', 'meta', 'brk_b', 'unh', 'ma', 'hd', 'pg', 'cost', 'dis', 'intc']

  Saved → ../data/stocks_log_returns.csv


/var/folders/vj/_0bszh_n0psbn0345y9vlrs40000gn/T/ipykernel_30825/1467923041.py:65: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_clean["Date"] = merged_clean["Date"].dt.strftime("%Y-%m-%d")


,Date,aapl,msft,nvda,amzn,jpm,jnj,xom,tsla,nflx,...,googl,meta,brk_b,unh,ma,hd,pg,cost,dis,intc
348,2012-05-21,0.056626,0.016266,0.017234,0.019725,-0.029699,0.001893,0.006972,0.042968,0.025124,...,0.022578,-0.116378,0.011216,0.028485,0.039799,0.011832,-0.002049,0.009480,0.013152,0.003064
349,2012-05-22,-0.007708,0.000337,-0.012280,-0.012828,0.045107,0.000787,-0.001098,0.068181,-0.057814,...,-0.021912,-0.093255,-0.001881,0.003056,0.006942,0.013560,-0.003793,-0.004189,0.000000,-0.004599
350,2012-05-23,0.024107,-0.022084,0.024411,0.009015,0.007323,-0.003943,0.001098,0.007118,0.060182,...,0.014311,0.031749,0.001255,-0.006482,0.008790,0.009897,-0.012108,0.002464,-0.004063,-0.022927
351,2012-05-24,-0.009227,-0.001375,-0.026886,-0.009433,-0.008500,0.006997,0.006924,-0.024145,-0.023071,...,-0.009562,0.031680,0.000627,0.015415,0.006547,0.019706,0.002881,0.013946,0.005189,0.008221
352,2012-05-25,-0.005374,-0.000344,0.023665,-0.010978,-0.013933,-0.009394,-0.006436,-0.015644,-0.000712,...,-0.020299,-0.034497,-0.006916,-0.001780,-0.014080,-0.005446,-0.001279,0.000000,0.001349,0.003502
